In [1]:
import pandas as pd
import numpy as np

# 1. Fetch from a rock-solid, evergreen data repository
url = "https://raw.githubusercontent.com/ageron/handson-ml/master/datasets/housing/housing.csv"
df_house = pd.read_csv(url)

print("--- Dataset Structural Audit ---")
print(f"Dataset Shape: {df_house.shape}")
print("\n--- First 5 Rows Preview ---")
print(df_house.head())


--- Dataset Structural Audit ---
Dataset Shape: (20640, 10)

--- First 5 Rows Preview ---
   longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0    -122.23     37.88                41.0        880.0           129.0   
1    -122.22     37.86                21.0       7099.0          1106.0   
2    -122.24     37.85                52.0       1467.0           190.0   
3    -122.25     37.85                52.0       1274.0           235.0   
4    -122.25     37.85                52.0       1627.0           280.0   

   population  households  median_income  median_house_value ocean_proximity  
0       322.0       126.0         8.3252            452600.0        NEAR BAY  
1      2401.0      1138.0         8.3014            358500.0        NEAR BAY  
2       496.0       177.0         7.2574            352100.0        NEAR BAY  
3       558.0       219.0         5.6431            341300.0        NEAR BAY  
4       565.0       259.0         3.8462            342200.0    

In [2]:
# 1. Look at the scale differences (Min vs Max)
print("--- Scale Mismatch Audit ---")
print(df_house[['median_income', 'total_rooms']].describe().loc[['min', 'max']])

# 2. Check for hidden missing values
print("\n--- Missing Value Audit ---")
print(df_house.isnull().sum())

# 3. Look at the text column categories
print("\n--- Categorical Text Audit ---")
print(df_house['ocean_proximity'].value_counts())

--- Scale Mismatch Audit ---
     median_income  total_rooms
min         0.4999          2.0
max        15.0001      39320.0

--- Missing Value Audit ---
longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64

--- Categorical Text Audit ---
ocean_proximity
<1H OCEAN     9136
INLAND        6551
NEAR OCEAN    2658
NEAR BAY      2290
ISLAND           5
Name: count, dtype: int64


In [3]:
# 1. Calculate the median of the total_bedrooms column
median_bedrooms = df_house['total_bedrooms'].median()
print(f"The calculated median bedroom count is: {median_bedrooms}")

# 2. Fill the missing values with that median
df_house['total_bedrooms'] = df_house['total_bedrooms'].fillna(median_bedrooms)

# 3. Double-check to ensure the 207 holes are completely gone
print("Missing values remaining in total_bedrooms:", df_house['total_bedrooms'].isnull().sum())

The calculated median bedroom count is: 435.0
Missing values remaining in total_bedrooms: 0


In [4]:
# 4. Use pandas get_dummies to convert text categories into binary columns (0 or 1)
df_house_encoded = pd.get_dummies(df_house, columns=['ocean_proximity'], dtype=int)

# 5. Look at the new columns we created
print("--- New Encoded Dataset Columns ---")
print(df_house_encoded.columns)

print("\n--- Preview of the newly transformed columns ---")
print(df_house_encoded.filter(like='ocean_proximity').head())

--- New Encoded Dataset Columns ---
Index(['longitude', 'latitude', 'housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income',
       'median_house_value', 'ocean_proximity_<1H OCEAN',
       'ocean_proximity_INLAND', 'ocean_proximity_ISLAND',
       'ocean_proximity_NEAR BAY', 'ocean_proximity_NEAR OCEAN'],
      dtype='object')

--- Preview of the newly transformed columns ---
   ocean_proximity_<1H OCEAN  ocean_proximity_INLAND  ocean_proximity_ISLAND  \
0                          0                       0                       0   
1                          0                       0                       0   
2                          0                       0                       0   
3                          0                       0                       0   
4                          0                       0                       0   

   ocean_proximity_NEAR BAY  ocean_proximity_NEAR OCEAN  
0                         1    

In [5]:
# 1. Calculate rooms per household
df_house_encoded['rooms_per_household'] = df_house_encoded['total_rooms'] / df_house_encoded['households']

# 2. Calculate bedrooms per room (Ratio of bedrooms to total rooms)
df_house_encoded['bedrooms_per_room'] = df_house_encoded['total_bedrooms'] / df_house_encoded['total_rooms']

# 3. Calculate population density per household
df_house_encoded['population_per_household'] = df_house_encoded['population'] / df_house_encoded['households']

# 4. Take a look at our brand new engineered columns
print("--- Engineered Features Preview ---")
print(df_house_encoded[['rooms_per_household', 'bedrooms_per_room', 'population_per_household']].head())

print(f"\nNew Dataset Shape: {df_house_encoded.shape}")

--- Engineered Features Preview ---
   rooms_per_household  bedrooms_per_room  population_per_household
0             6.984127           0.146591                  2.555556
1             6.238137           0.155797                  2.109842
2             8.288136           0.129516                  2.802260
3             5.817352           0.184458                  2.547945
4             6.281853           0.172096                  2.181467

New Dataset Shape: (20640, 17)


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Separate our features (X) from the price label we want to predict (y) along with dropping irrelevant columns
X = df_house_encoded.drop(['median_house_value','total_rooms', 'total_bedrooms', 'population', 'households'], axis=1)
y = df_house_encoded['median_house_value']

# 2. Split into 80% training data and 20% testing data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Initialize the standardizer
scaler = StandardScaler()

# 4. Calculate scaling parameters using ONLY training data, then transform both
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("--- Preprocessing Pipeline Complete ---")
print(f"Cleaned Training Features Shape: {X_train_scaled.shape}")
print(f"Cleaned Testing Features Shape: {X_test_scaled.shape}")

--- Preprocessing Pipeline Complete ---
Cleaned Training Features Shape: (16512, 12)
Cleaned Testing Features Shape: (4128, 12)


In [7]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# 1. Initialize the Random Forest Regressor optimized for your laptop hardware
model_house = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)

# 2. Train the model on our pristine, streamlined feature deck
print("Training the housing regression model... (This may take 10-15 seconds)")
model_house.fit(X_train_scaled, y_train)

# 3. Predict the house prices for our test dataset
y_pred = model_house.predict(X_test_scaled)

# 4. Calculate the performance scorecards
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\n--- Model Evaluation Completed ---")
print(f"Root Mean Squared Error (RMSE): ${rmse:,.2f}")
print(f"R-squared (R2) Score: {r2:.4f}")

Training the housing regression model... (This may take 10-15 seconds)

--- Model Evaluation Completed ---
Root Mean Squared Error (RMSE): $50,452.15
R-squared (R2) Score: 0.8058


In [8]:
import joblib

# 1. Save the trained model to a file
joblib.dump(model_house, 'housing_model.pkl')

# 2. Save the scaler (so the web app can scale new data exactly the same way)
joblib.dump(scaler, 'scaler.pkl')

print("Model and Scaler successfully saved to your hard drive!")

Model and Scaler successfully saved to your hard drive!


In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
import joblib
from google.colab import files

# 1. Classify features matching your training DataFrame columns
categorical_features = ['ocean_proximity']
numerical_features = [
    'MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 
    'Population', 'Households', 'Latitude', 'Longitude'
]

# 2. Setup the categorical column transformer matrix
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ],
    remainder='passthrough'
)

# 3. Securely bind the encoder and the Random Forest engine block
housing_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42))
])

# 4. Fit the complete operational pipeline (Replace X_train/y_train with your variables)
housing_pipeline.fit(X_train[categorical_features + numerical_features], y_train)

# 5. Export and trigger the local file download
joblib.dump(housing_pipeline, 'housing_model.pkl')
files.download('housing_model.pkl')

print("Unified housing pipeline exported! Move this file into D:\\coding\\web\\ai_portfolio\\models\\")

NameError: name 'X_train' is not defined